# KNN Regression

While KNN is most commonly introduced as a classification algorithm, it works equally well for regression tasks. Instead of a majority vote, KNN regression predicts the **average** (or weighted average) of the target values of the K nearest neighbors. This notebook explores KNN regression, the effect of K on smoothness, weighted variants, and comparisons with parametric models.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_regression

np.random.seed(42)

---
## 1. KNN for Regression: Concept

Given a query point $x_q$, KNN regression:

1. Finds the K training points closest to $x_q$.
2. Computes the prediction as the **mean** of their target values:

$$\hat{y}(x_q) = \frac{1}{K} \sum_{i \in \mathcal{N}_K(x_q)} y_i$$

where $\mathcal{N}_K(x_q)$ is the set of K nearest neighbors.

This is the simplest non-parametric regression method: no functional form is assumed, and the model adapts entirely to the local structure of the data.

In [ ]:
# Generate a simple 1D non-linear function with noise
X_1d = np.sort(np.random.uniform(0, 10, 100)).reshape(-1, 1)
y_true_func = np.sin(X_1d).ravel() + 0.5 * np.cos(2 * X_1d).ravel()
y_1d = y_true_func + np.random.normal(0, 0.3, size=len(X_1d))

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(X_1d, y_1d, c='steelblue', s=20, alpha=0.6, label='Noisy samples')
ax.plot(X_1d, y_true_func, 'k--', linewidth=1.5, label='True function')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Synthetic 1D Regression Data')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. KNeighborsRegressor in Scikit-Learn

The API mirrors `KNeighborsClassifier`:

| Parameter | Default | Description |
|-----------|---------|-------------|
| `n_neighbors` | 5 | Number of neighbors (K) |
| `weights` | `'uniform'` | `'uniform'` (simple average) or `'distance'` (weighted by inverse distance) |
| `metric` | `'minkowski'` | Distance metric |
| `p` | 2 | Minkowski power parameter |

In [ ]:
# Fit a KNN regressor with K=5
knn_reg = KNeighborsRegressor(n_neighbors=5)
knn_reg.fit(X_1d, y_1d)

# Predict on a fine grid for smooth visualization
X_grid = np.linspace(0, 10, 500).reshape(-1, 1)
y_pred_grid = knn_reg.predict(X_grid)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(X_1d, y_1d, c='steelblue', s=20, alpha=0.5, label='Training data')
ax.plot(X_grid, y_pred_grid, 'r-', linewidth=2, label='KNN (K=5)')
ax.plot(X_1d, y_true_func, 'k--', linewidth=1, alpha=0.5, label='True function')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('KNN Regression (K=5)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on the training data (for illustration)
y_pred_train = knn_reg.predict(X_1d)
print(f"Training MSE:  {mean_squared_error(y_1d, y_pred_train):.4f}")
print(f"Training R^2:  {r2_score(y_1d, y_pred_train):.4f}")

---
## 3. Weighted KNN: weights='distance'

With uniform weights, all K neighbors contribute equally. With `weights='distance'`, closer neighbors receive more influence:

$$\hat{y}(x_q) = \frac{\sum_{i \in \mathcal{N}_K} \frac{1}{d(x_q, x_i)} \cdot y_i}{\sum_{i \in \mathcal{N}_K} \frac{1}{d(x_q, x_i)}}$$

This often improves predictions in regions where the density of training points is uneven.

In [ ]:
# Compare uniform vs distance weights
knn_uniform = KNeighborsRegressor(n_neighbors=7, weights='uniform')
knn_distance = KNeighborsRegressor(n_neighbors=7, weights='distance')

knn_uniform.fit(X_1d, y_1d)
knn_distance.fit(X_1d, y_1d)

y_uniform = knn_uniform.predict(X_grid)
y_distance = knn_distance.predict(X_grid)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, y_hat, title in zip(axes,
                             [y_uniform, y_distance],
                             ["Uniform Weights", "Distance Weights"]):
    ax.scatter(X_1d, y_1d, c='steelblue', s=20, alpha=0.5)
    ax.plot(X_grid, y_hat, 'r-', linewidth=2, label='KNN prediction')
    ax.plot(X_1d, y_true_func, 'k--', linewidth=1, alpha=0.5, label='True function')
    ax.set_title(f'K=7, {title}')
    ax.set_xlabel('x')
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('y')
plt.tight_layout()
plt.show()

print(f"Uniform  - Training MSE: {mean_squared_error(y_1d, knn_uniform.predict(X_1d)):.4f}")
print(f"Distance - Training MSE: {mean_squared_error(y_1d, knn_distance.predict(X_1d)):.4f}")

Distance-weighted predictions tend to be smoother near the training points and can better capture local patterns.

---
## 4. Regression Example on Synthetic Data

Let's use `make_regression` to create a multi-feature dataset and evaluate KNN regression with a proper train/test split.

In [ ]:
# Multi-feature synthetic regression
X_reg, y_reg = make_regression(
    n_samples=500, n_features=5, n_informative=3,
    noise=20.0, random_state=42
)

X_r_train, X_r_test, y_r_train, y_r_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=42
)

print(f"Training set: {X_r_train.shape}")
print(f"Test set:     {X_r_test.shape}")

In [ ]:
# Build a pipeline with scaling + KNN regression
pipe_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(n_neighbors=7, weights='distance'))
])

pipe_reg.fit(X_r_train, y_r_train)
y_r_pred = pipe_reg.predict(X_r_test)

print(f"Test MSE: {mean_squared_error(y_r_test, y_r_pred):.2f}")
print(f"Test R^2: {r2_score(y_r_test, y_r_pred):.4f}")

In [ ]:
# Predicted vs actual scatter plot
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_r_test, y_r_pred, alpha=0.5, edgecolors='k', s=30)

# Perfect prediction line
lims = [min(y_r_test.min(), y_r_pred.min()), max(y_r_test.max(), y_r_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')

ax.set_xlabel('Actual')
ax.set_ylabel('Predicted')
ax.set_title('KNN Regression: Predicted vs Actual')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Visualizing Predictions vs the True Function

Returning to our 1D example, we can directly overlay the KNN prediction curve against the underlying noise-free function to see how well KNN recovers the true signal.

In [ ]:
# Split the 1D data
X_1d_tr, X_1d_te, y_1d_tr, y_1d_te = train_test_split(
    X_1d, y_1d, test_size=0.25, random_state=42
)

# Fit with K=7, distance weights
knn_1d = KNeighborsRegressor(n_neighbors=7, weights='distance')
knn_1d.fit(X_1d_tr, y_1d_tr)

y_grid_pred = knn_1d.predict(X_grid)
y_true_grid = np.sin(X_grid).ravel() + 0.5 * np.cos(2 * X_grid).ravel()

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(X_1d_tr, y_1d_tr, c='steelblue', s=25, alpha=0.5, label='Train data')
ax.scatter(X_1d_te, y_1d_te, c='orange', s=25, alpha=0.7, marker='^', label='Test data')
ax.plot(X_grid, y_grid_pred, 'r-', linewidth=2, label='KNN prediction')
ax.plot(X_grid, y_true_grid, 'k--', linewidth=1.5, label='True function')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('KNN Regression: Predictions vs True Function')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

test_mse = mean_squared_error(y_1d_te, knn_1d.predict(X_1d_te))
print(f"Test MSE: {test_mse:.4f}")

---
## 6. Effect of K on Smoothness

K controls how many neighbors are averaged, which directly affects the smoothness of the prediction curve:

- **K=1**: The prediction passes through every training point (zero training error), producing a jagged, overfitting curve.
- **Large K**: The prediction averages over many neighbors, producing a smooth but potentially underfit curve.

In [ ]:
k_values_reg = [1, 3, 10, 40]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

for ax, k in zip(axes.ravel(), k_values_reg):
    knn_k = KNeighborsRegressor(n_neighbors=k)
    knn_k.fit(X_1d, y_1d)
    y_k = knn_k.predict(X_grid)

    ax.scatter(X_1d, y_1d, c='steelblue', s=15, alpha=0.4)
    ax.plot(X_grid, y_k, 'r-', linewidth=2, label=f'KNN (K={k})')
    ax.plot(X_1d, y_true_func, 'k--', linewidth=1, alpha=0.5, label='True function')
    ax.set_title(f'K = {k}')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

axes[1, 0].set_xlabel('x')
axes[1, 1].set_xlabel('x')
axes[0, 0].set_ylabel('y')
axes[1, 0].set_ylabel('y')

plt.suptitle('Effect of K on Smoothness in KNN Regression', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Quantify the bias-variance trade-off with K
k_sweep = range(1, 51)
train_mses = []
test_mses = []

for k in k_sweep:
    knn_k = KNeighborsRegressor(n_neighbors=k)
    knn_k.fit(X_1d_tr, y_1d_tr)
    train_mses.append(mean_squared_error(y_1d_tr, knn_k.predict(X_1d_tr)))
    test_mses.append(mean_squared_error(y_1d_te, knn_k.predict(X_1d_te)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(k_sweep), train_mses, 'o-', label='Train MSE', alpha=0.7)
ax.plot(list(k_sweep), test_mses, 's-', label='Test MSE', alpha=0.7)
ax.set_xlabel('K (Number of Neighbors)')
ax.set_ylabel('Mean Squared Error')
ax.set_title('KNN Regression: Train vs Test MSE')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_k_reg = list(k_sweep)[np.argmin(test_mses)]
print(f"Best K on test set: {best_k_reg} (MSE = {min(test_mses):.4f})")

K=1 has zero training error (the model memorizes every point) but high test error. As K increases, training error rises while test error initially decreases before eventually increasing due to underfitting.

---
## 7. Comparing KNN Regression to Linear Regression

Linear regression fits a global parametric model (a line or hyperplane) to the data. KNN fits locally with no global functional form. This comparison highlights where each approach excels.

In [ ]:
# Fit both models on the 1D data
lin_reg = LinearRegression()
lin_reg.fit(X_1d_tr, y_1d_tr)

knn_best = KNeighborsRegressor(n_neighbors=7, weights='distance')
knn_best.fit(X_1d_tr, y_1d_tr)

y_lin = lin_reg.predict(X_grid)
y_knn = knn_best.predict(X_grid)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(X_1d_tr, y_1d_tr, c='steelblue', s=20, alpha=0.4, label='Training data')
ax.scatter(X_1d_te, y_1d_te, c='orange', s=25, alpha=0.7, marker='^', label='Test data')
ax.plot(X_grid, y_knn, 'r-', linewidth=2, label='KNN (K=7, distance)')
ax.plot(X_grid, y_lin, 'g-', linewidth=2, label='Linear Regression')
ax.plot(X_grid, np.sin(X_grid).ravel() + 0.5 * np.cos(2 * X_grid).ravel(),
        'k--', linewidth=1.5, label='True function')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('KNN vs Linear Regression')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Numerical comparison
results = {
    'Model': ['Linear Regression', 'KNN (K=7, distance)'],
    'Train MSE': [
        mean_squared_error(y_1d_tr, lin_reg.predict(X_1d_tr)),
        mean_squared_error(y_1d_tr, knn_best.predict(X_1d_tr)),
    ],
    'Test MSE': [
        mean_squared_error(y_1d_te, lin_reg.predict(X_1d_te)),
        mean_squared_error(y_1d_te, knn_best.predict(X_1d_te)),
    ],
    'Test R^2': [
        r2_score(y_1d_te, lin_reg.predict(X_1d_te)),
        r2_score(y_1d_te, knn_best.predict(X_1d_te)),
    ],
}

for i in range(len(results['Model'])):
    print(f"{results['Model'][i]:30s}  "
          f"Train MSE: {results['Train MSE'][i]:8.4f}  "
          f"Test MSE: {results['Test MSE'][i]:8.4f}  "
          f"Test R^2: {results['Test R^2'][i]:7.4f}")

Linear regression draws a straight line through the data and cannot capture the non-linear pattern. KNN, being non-parametric, adapts to the local curvature and achieves much lower error on this dataset.

---
## 8. When to Use KNN vs Parametric Models

| Criterion | KNN | Parametric (e.g., Linear Regression) |
|-----------|-----|--------------------------------------|
| **Assumptions** | None about data distribution | Assumes a functional form (linear, polynomial, etc.) |
| **Non-linear relationships** | Captures automatically | Requires explicit feature engineering or non-linear models |
| **High-dimensional data** | Struggles (curse of dimensionality) | Often handles well with regularization |
| **Large training sets** | Slow prediction (must scan all points) | Fast prediction (fixed-size parameter vector) |
| **Small datasets** | Can work well if features are few | May overfit or underfit depending on model complexity |
| **Interpretability** | Limited (black-box local averaging) | High (coefficients have meaning) |

**Rules of thumb:**
- Use KNN when the relationship is non-linear, the feature space is low-dimensional, and the dataset is not too large.
- Use parametric models when you need interpretability, fast prediction, or when the data is high-dimensional.

In [ ]:
# Cross-validate KNN regression on the multi-feature dataset
k_range_cv = [1, 3, 5, 7, 10, 15, 20, 30]

print(f"{'K':>4s}  {'Mean CV R^2':>12s}  {'Std':>8s}")
print('-' * 30)

for k in k_range_cv:
    pipe_cv = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsRegressor(n_neighbors=k, weights='distance'))
    ])
    scores = cross_val_score(pipe_cv, X_reg, y_reg, cv=5, scoring='r2')
    print(f"{k:4d}  {scores.mean():12.4f}  {scores.std():8.4f}")

---
## 9. Summary: KNN Strengths and Weaknesses

### Strengths

- **Simple and intuitive** — predictions are just averages of nearby values.
- **No training phase** — model stores the data and computes at prediction time.
- **No assumptions** — non-parametric; adapts to any underlying function.
- **Works for both classification and regression** with the same core algorithm.

### Weaknesses

- **Slow prediction** — O(n * d) per query; impractical for large datasets without approximate nearest neighbor techniques (e.g., KD-trees, ball trees).
- **Memory intensive** — must store all training data.
- **Curse of dimensionality** — performance degrades as the number of features grows.
- **Sensitive to feature scaling** — features on different scales distort distances.
- **Cannot extrapolate** — predictions outside the training range default to the nearest boundary points, so KNN fails on extrapolation tasks.

In [ ]:
# Demonstrate the extrapolation problem
X_extra = np.linspace(-2, 14, 500).reshape(-1, 1)  # extends beyond training range [0, 10]
y_extra_true = np.sin(X_extra).ravel() + 0.5 * np.cos(2 * X_extra).ravel()

knn_extrap = KNeighborsRegressor(n_neighbors=5)
knn_extrap.fit(X_1d, y_1d)
y_extra_pred = knn_extrap.predict(X_extra)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(X_1d, y_1d, c='steelblue', s=20, alpha=0.4, label='Training data')
ax.plot(X_extra, y_extra_true, 'k--', linewidth=1.5, label='True function')
ax.plot(X_extra, y_extra_pred, 'r-', linewidth=2, label='KNN prediction')
ax.axvspan(-2, 0, alpha=0.1, color='red', label='Extrapolation zone')
ax.axvspan(10, 14, alpha=0.1, color='red')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('KNN Cannot Extrapolate Beyond the Training Range')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In the shaded regions, KNN predictions flatten out because there are no training points beyond the range. The model simply returns the average of the nearest boundary points. This is a fundamental limitation of instance-based methods.

---
## Key Takeaways

1. **KNN regression** predicts by averaging the target values of the K nearest neighbors.
2. **Weighted KNN** (`weights='distance'`) gives closer neighbors more influence and often improves results.
3. **K controls smoothness** — small K is jagged (overfitting), large K is smooth (underfitting).
4. **KNN excels** at capturing non-linear relationships without requiring explicit feature engineering.
5. **KNN struggles** with high-dimensional data, large datasets, and extrapolation.
6. Always **scale features** before using KNN, and use **cross-validation** to select K.